# Airbnb Price Prediction & Review Analysis

This project analyzes Las Vegas Airbnb listings using data from [Inside Airbnb](https://insideairbnb.com/get-the-data/) to answer two questions:

1. **What features drive listing prices?**
2. **What do guests actually talk about in their reviews?**

## Price Prediction

A neural network (TensorFlow/Keras) trained on listing attributes — bedrooms, bathrooms, room type, host attributes, and neighborhood. Prices were capped at $1,000/night to scope the model to standard rentals (excluding luxury party homes, a distinct market segment). Test set performance:

- **R²: 0.504** — explains roughly half of price variance from listing features alone
- **MAE: 58.50** - average prediction price is within ~$60 of actual price
- **Key finding:** capacity (bathrooms, bedrooms) drives most predictions; location features contribute surprisingly little once room type is known

## Review Topic Modeling

NLP analysis of 10,000 guest reviews using four topic modeling approaches — LDA, NMF, LSA (traditional bag-of-words methods), and BERTopic (transformer-based embeddings). Reviews were filtered to English only before analysis. Consistent themes across methods: location relative to the Strip, cleanliness, host hospitality, and overall comfort.

**Tools & Libraries:** Python, pandas, NumPy, scikit-learn, TensorFlow/Keras, NLTK, BERTopic, Seaborn, Matplotlib, Plotly

*Group project for Applications of AI Models at UMass Amherst (Spring 2026)*

## Data Loading and Exploration

In [ ]:
!pip install bertopic
!pip install langdetect

In [ ]:
# Standard library
import re
import string

# Data & numerical
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from wordcloud import WordCloud

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Topic modeling — traditional
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation, NMF, TruncatedSVD, PCA

# Topic modeling — transformer-based
from bertopic import BERTopic

# Utilities
from tqdm import tqdm
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 42  # makes detection reproducible

# Modeling
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
# Download required NLTK data
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('punkt_tab')

In [ ]:
# import datasets
listings_url = 'https://data.insideairbnb.com/united-states/nv/clark-county-nv/2025-09-23/data/listings.csv.gz'
reviews_url = 'https://data.insideairbnb.com/united-states/nv/clark-county-nv/2025-09-23/data/reviews.csv.gz'

# Load the datasets into DataFrames
listings_df = pd.read_csv(listings_url, compression='gzip')
reviews_df = pd.read_csv(reviews_url, compression='gzip')

In [ ]:
reviews_df.head()

In [ ]:
reviews_df.info()

In [ ]:
reviews_df.isnull().sum()

In [ ]:
listings_df.head()

In [ ]:
listings_df.info()

In [ ]:
listings_df.isnull().sum()

In [ ]:
listings_df['price'].isnull().sum()

### Exploratory Data Analysis

In [ ]:
#Print all unique host locations in the dataset
print(listings_df['host_location'].unique())      # List all location names
print(len(listings_df['host_location'].unique())) # Total number of unique locations

In [ ]:
#Filter Listings for Vegas
vegas_df = listings_df[listings_df['host_location'] == 'Las Vegas, NV'].copy()
print(vegas_df.shape)

Below we converted price from string "$150" to numeric, so that modeling and visualizations can happen.

In [ ]:
vegas_df['price'] = vegas_df['price'].replace(r'[\$,]', '', regex=True).astype(float)

In [ ]:
vegas_df.head()

In [ ]:
#Define the columns we want to keep and create a new dataframe with the selected columns
selected_columns = [
    'price', 'bathrooms','bedrooms', 'number_of_reviews', 'latitude',
    'longitude', 'room_type', 'host_identity_verified', 'host_is_superhost', 'neighbourhood'
]

vegas_df = vegas_df[selected_columns]
vegas_df.head()

In [ ]:
vegas_df.isnull().sum()

Filled N/A values from bathroom and bedroom with the median from the dataset, ensuring we keep valuable data points in the model.

In [ ]:
vegas_df['bathrooms'] = vegas_df['bathrooms'].fillna(vegas_df['bathrooms'].median())
vegas_df['bedrooms'] = vegas_df['bedrooms'].fillna(vegas_df['bedrooms'].median())

In [ ]:
vegas_df.isnull().sum()

Filled N/A superhost with 'f' for False, and filled N/A neighbourhood with 'Unknown' to keep data neat.

In [ ]:
vegas_df['host_is_superhost'] = vegas_df['host_is_superhost'].fillna('f')
vegas_df['neighbourhood'] = vegas_df['neighbourhood'].fillna('Unknown')

In [ ]:
vegas_df.isnull().sum()

Cap at $1000 a night to get rid of extreme outliers and keep data reliable

In [ ]:
# Cap at $1000/night — excludes luxury party rentals which are a distinct market
PRICE_CAP = 1000
vegas_df = vegas_df[vegas_df['price'] <= PRICE_CAP]
print(f"Removed {((vegas_df['price'] > PRICE_CAP).sum())} listings above ${PRICE_CAP}")
print(f"Working with {len(vegas_df)} listings")

One hot encode categorical variables to convert them to numeric format for modeling

In [ ]:
#With room_type being turned into a dummy variable the box plot doesn't work,
#so a copy of the dataframe from before the dummy variables is used for the boxplot
vegas_df_plot = vegas_df.copy()

In [ ]:
vegas_df = pd.get_dummies(vegas_df, columns=[
    'room_type',
    'host_identity_verified',
    'host_is_superhost',
    'neighbourhood'
], drop_first=True)

Price distribution, heavily right skewed

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(vegas_df['price'], bins=50)

plt.xlim(0, 2000)
plt.title("Price Distribution (Vegas, Filtered View)")
plt.xlabel("Price")
plt.ylabel("Count")

plt.show()

Below, we have a scatterplot to show correlation between number of bedrooms and price

In [ ]:
sns.scatterplot(x='bedrooms', y='price', data=vegas_df)
plt.show()

Correlation matrix

In [ ]:
plt.figure(figsize=(10,8))
sns.heatmap(
    vegas_df.corr(),
    cmap='coolwarm',
    annot=True,
    fmt=".2f",   # rounds to 2 decimals
    linewidths=0.5
)
plt.title("Correlation Matrix")
plt.show()

Box plot for different room type price distributions.

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(x='room_type', y='price', data=vegas_df_plot)

plt.ylim(0, 1000)
plt.xticks(rotation=30)
plt.title("Price by Room Type")
plt.show()

In [ ]:
# Geospatial Plot

# Create price buckets
vegas_df['price_range'] = pd.cut(
    vegas_df['price'],
    bins=[0, 100, 200, 300, vegas_df['price'].max()],
    labels=['< $100', '$100–$200', '$200–$300', '>$300']
)

# Plot the interactive map
fig = px.scatter_mapbox(
    vegas_df,
    lat='latitude',
    lon='longitude',
    color='price_range',
    hover_data=['price', 'bedrooms', 'bathrooms'],
    zoom=10,
    mapbox_style="open-street-map",
    title="Airbnb Listings in Las Vegas by Price Range"
)

fig.show()

Summary Statistics

In [ ]:
vegas_df[['price','bedrooms','bathrooms','number_of_reviews']].describe()

## Price Prediction with Neural Network

We build a feed-forward neural network using TensorFlow/Keras to predict listing prices from listing attributes (bedrooms, bathrooms, room type, host attributes, neighborhood, location).

The workflow follows four steps:
1. **Data preparation** — handle target leakage, split, and scale
2. **Model architecture and training** — build the network and fit with early stopping
3. **Evaluation** — measure performance on the held-out test set
4. **Interpretation** — identify which features drive predictions

### Step 1: Data Preparation

Before training, three preparation steps:

- **Drop `price_range`** — this column was created from `price` for the geospatial plot, so leaving it in would leak the target into the model
- **Cast boolean dummy columns to integers** — TensorFlow handles numeric inputs more cleanly than booleans
- **Separate features (X) from target (y)**, then 80/20 train-test split

We also **standardize the features** with `StandardScaler`. Neural networks are sensitive to feature scale. Without scaling, features with large magnitudes (like latitude) dominate the gradient updates over 0/1 dummy columns. The scaler is fit on training data only and then applied to test data, to avoid information leaking from test into training.

In [ ]:
# Drop price_range (created from price for the map — would leak the target into the model)
vegas_model_df = vegas_df.drop(columns=['price_range'])

# Convert any boolean dummy columns to int (cleaner for TF)
vegas_model_df = vegas_model_df.astype({col: 'int' for col in vegas_model_df.select_dtypes('bool').columns})

# Separate features and target
X = vegas_model_df.drop(columns=['price'])
y = vegas_model_df['price']

print(f"Features: {X.shape[1]} columns, {X.shape[0]} rows")
print(f"Target range: ${y.min():.0f} – ${y.max():.0f}")

In [ ]:
# 80/20 train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Standardize features — fit only on training set to avoid data leakage
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set: {X_train_scaled.shape}")
print(f"Test set:     {X_test_scaled.shape}")

### Step 2: Build and Train the Neural Network

**Architecture:** A small neural network with two hidden layers (64 and 32 neurons, ReLU activation), each followed by 20% dropout for regularization, and a single linear output for the regression target. Adam optimizer with mean squared error loss.

**Early stopping:** During training, we monitor validation loss and halt when it stops improving for 10 consecutive epochs, then restore the best weights seen. This prevents overfitting and saves time — the model stops once it's done learning useful patterns rather than running a fixed number of epochs.

In [ ]:
# Feed-forward neural network for regression
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(1)  # linear output for regression
])

model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

model.summary()

In [ ]:
# Early stopping to prevent overfitting by halting when validation loss stops improving
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train_scaled, y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=0
)

print(f"Training stopped at epoch {len(history.history['loss'])}")

### Step 3: Evaluate Model Performance

**Training history.** The loss and MAE curves below should both decrease and plateau without diverging. One thing to note: validation loss may appear slightly *lower* than training loss. This is an expected artifact of dropout, which is active during training (handicapping the network) but disabled during validation.

**Test set metrics.** Three regression metrics, each telling us something different:
- **MAE** (Mean Absolute Error) — average dollar miss, easy to interpret
- **RMSE** (Root Mean Squared Error) — penalizes large errors more heavily; comparing MAE to RMSE reveals whether errors are evenly spread or driven by a few big misses
- **R²** — fraction of price variance captured by the model

**Predicted vs Actual plot.** Each point is a test-set listing. Predictions falling on the red dashed line are perfect; the spread around the line shows where the model performs well versus where it struggles.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['loss'], label='Train')
axes[0].plot(history.history['val_loss'], label='Validation')
axes[0].set_title('Loss (MSE) over Epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE')
axes[0].legend()

axes[1].plot(history.history['mae'], label='Train')
axes[1].plot(history.history['val_mae'], label='Validation')
axes[1].set_title('Mean Absolute Error over Epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE ($)')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Predict on the held-out test set
y_pred = model.predict(X_test_scaled, verbose=0).flatten()

# Compute regression metrics
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"Test MAE:  ${mae:.2f}")
print(f"Test RMSE: ${rmse:.2f}")
print(f"Test R²:   {r2:.3f}")

In [ ]:
plt.figure(figsize=(7, 7))
plt.scatter(y_test, y_pred, alpha=0.4, s=20)
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()],
         'r--', lw=2, label='Perfect prediction')
plt.xlabel('Actual Price ($)')
plt.ylabel('Predicted Price ($)')
plt.title('Actual vs Predicted Prices (Test Set)')
plt.legend()
plt.tight_layout()
plt.show()

### Step 4: Feature Importance via Permutation

Neural networks don't expose feature importance directly the way linear models or tree-based models do. Instead, we use **permutation importance**: for each feature, we shuffle its values in the test set and measure how much R² drops. Larger drops mean the model relies more on that feature. This gives us a model-agnostic way to interpret which inputs are doing the predictive work.

In [ ]:
# Neural networks don't expose feature importance directly. Permutation importance
# shuffles each feature and measures how much performance drops — bigger drop = more important.

def r2_scorer(estimator, X, y):
    preds = estimator.predict(X, verbose=0).flatten()
    return r2_score(y, preds)

perm_result = permutation_importance(
    model, X_test_scaled, y_test,
    n_repeats=10, random_state=42, scoring=r2_scorer
)

importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance': perm_result.importances_mean,
    'std': perm_result.importances_std
}).sort_values('importance', ascending=False)

# Plot top 15
top_n = 15
plt.figure(figsize=(10, 6))
top = importance_df.head(top_n).iloc[::-1]
plt.barh(top['feature'], top['importance'], xerr=top['std'])
plt.xlabel('Permutation Importance (drop in R²)')
plt.title(f'Top {top_n} Features Driving Price Predictions')
plt.tight_layout()
plt.show()

print(f"Top {top_n} features:")
print(importance_df.head(top_n).to_string(index=False))

## Review Analysis & Topic Modeling


### Step 1: Load Data and Text Preprocessing

Before topic modeling can be applied, the raw review text needs to be cleaned and
converted into a numeric format that models can process.

The preprocessing pipeline includes:
- Lowercasing and removing punctuation and URLs
- Tokenizing text into individual words
- Removing stopwords (common words like "the", "and", "is" that carry no meaning)  
- Lemmatizing words to their root form (e.g., "running" -> "run")
- Building **Bag-of-Words (BoW)** and **TF-IDF** matrix representations

BoW captures raw word frequency, while TF-IDF downweights words that appear
across many reviews, highlighting more distinctive terms. Both representations
are used as input for the traditional topic models (LDA, NMF, and LSA) in the
next step.

**1.1 Import Libraries and Load Data**

In [ ]:
# Display the first few rows of each dataset
reviews_df.head()


In [ ]:
#check the shape of the dataframe
reviews_df.shape

**1.2 Text Cleaning Function**

We’ll define a reusable function that:
- Removes URLs, punctuation, and numbers
- Convert to lower case  
- Tokenizes text and removes stopwords  
- Lemmatizes each word to its root form

This produces cleaner, more consistent text for modeling.

In [ ]:
# pre- processing function

# Initialize lemmatizer
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    if pd.isnull(text):
        return ''
    # Remove URLs, non-letters, and convert to lowercase
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = text.lower()

    # Tokenize and remove stopwords - word_tokenize() would be more accurate but slow
    tokens = text.split()
    stop_words = set(stopwords.words('english'))

    # Lemmatize and remove stopwords
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words and word not in string.punctuation]

    return ' '.join(tokens)

**1.3 Filter and Preprocess Reviews**

We now apply the `preprocess_text()` function to clean each review comment.  
To avoid performance issues, we’ll work with a **sample of 10,000 reviews** from 2025.  
This keeps our analysis representative but much faster to run.

> ⚠️ Note: We use `.copy()` to avoid the `SettingWithCopyWarning` from pandas when modifying sliced data.



In [ ]:
# Ensure the 'date' column is in datetime format
reviews_df['date'] = pd.to_datetime(reviews_df['date'], errors='coerce')

# Filter to 2025 reviews
filtered_reviews = reviews_df[(reviews_df['date'].dt.year == 2025)].copy()
print(f"2025 reviews: {len(filtered_reviews)}")

# Sample down first — language detection is slow on the full set
filtered_reviews = filtered_reviews.sample(n=12000, random_state=42)

# Detect language on raw comments
def safe_detect(text):
    try:
        return detect(str(text))
    except:
        return 'unknown'

print("Detecting review languages...")
filtered_reviews['language'] = filtered_reviews['comments'].apply(safe_detect)
print(filtered_reviews['language'].value_counts().head(5))

# Keep only English
filtered_reviews = filtered_reviews[filtered_reviews['language'] == 'en']
print(f"After English filter: {len(filtered_reviews)}")

# Trim to exactly 10K (or whatever's left if less)
filtered_reviews = filtered_reviews.head(10000)

# Now preprocess
filtered_reviews['cleaned_comments'] = filtered_reviews['comments'].apply(preprocess_text)

In [ ]:
filtered_reviews.head()

### Step 2: Convert Text to Numeric Form (Bag-of-Words or TF-IDF)

**Vectorization Choices**

We represent each review as numeric features using **Bag-of-Words (BoW)** or **TF–IDF**.

- **BoW** favors frequent words (counts how often each term appears).  
- **TF–IDF** downweights ubiquitous terms that appear in many reviews, highlighting distinctive ones.

We’ll compare which representation yields more interpretable topics.

**Key Parameters**
- `min_df = 5` → ignore very rare tokens (appear in fewer than 5 documents)  
- `max_df = 0.9` → ignore very common tokens (appear in > 90 % of documents)  
- `stop_words = 'english'` → remove common English words (e.g., *the*, *and*, *is*)  
- `max_features = 5000` *(optional)* → cap the vocabulary to speed up modeling  
- `ngram_range = (1, 1)` → unigrams only (try `(1, 2)` later for unigrams + bigrams)




In [ ]:
# Vectorize the cleaned comments (Bag-of-Words)

vectorizer = CountVectorizer(
    min_df=5,
    max_df=0.90,
    stop_words='english',
    max_features=5000,       # <- keep models fast
    ngram_range=(1, 2)       # <- (1, 2) to include bigrams
)

X_bow = vectorizer.fit_transform(filtered_reviews['cleaned_comments'])
feature_names = vectorizer.get_feature_names_out()

print("BoW matrix shape:", X_bow.shape)



In [ ]:
# Vectorize the cleaned comments (TF-IDF)

tfidf_vectorizer = TfidfVectorizer(
    min_df=5,
    max_df=0.90,
    stop_words='english',
    max_features=5000,
    ngram_range=(1, 2)
)

# This creates the TF-IDF matrix directly
X_tfidf = tfidf_vectorizer.fit_transform(filtered_reviews['cleaned_comments'])
tfidf_feature_names = tfidf_vectorizer.get_feature_names_out()

print("TF-IDF matrix shape:", X_tfidf.shape)

In [ ]:
# Define functions to create plots


# Function to plot the top words for each topic
def plot_top_words(model, feature_names, n_top_words=10, title='Top words per topic'):
    fig, axes = plt.subplots(1, model.n_components, figsize=(15, 5), sharex=True)
    axes = axes.flatten()
    for topic_idx, topic in enumerate(model.components_):
        top_features_ind = topic.argsort()[:-n_top_words - 1:-1]
        top_features = [feature_names[i] for i in top_features_ind]
        weights = topic[top_features_ind]
        ax = axes[topic_idx]
        ax.barh(top_features, weights, height=0.7)
        ax.set_title(f'Topic {topic_idx+1}', fontdict={'fontsize': 10})
        ax.invert_yaxis()
        ax.tick_params(axis='both', which='major', labelsize=10)
    plt.suptitle(title, fontsize=12)
    plt.subplots_adjust(top=0.85, wspace=0.3)
    plt.show()


# Function to generate word clouds for each topic
def plot_word_clouds(model, feature_names, n_top_words=30):
    for topic_idx, topic in enumerate(model.components_):
        top_features_ind = topic.argsort()[:-n_top_words - 1:-1]
        top_features = {feature_names[i]: topic[i] for i in top_features_ind}
        wordcloud = WordCloud(width=800, height=400, background_color='white').generate_from_frequencies(top_features)
        plt.figure()
        plt.imshow(wordcloud, interpolation='bilinear')
        plt.axis('off')
        plt.title(f'Topic {topic_idx+1} Word Cloud', fontsize=16)
        plt.show()

### Step 3: Traditional Topic Modeling Approaches

We’ll begin with three classical approaches that use Bag-of-Words or TF-IDF representations.

### 1. Latent Dirichlet Allocation (LDA)
- Treats each document as a mixture of topics.  
- Each topic is represented as a distribution of words.  
- Helps discover *interpretable themes* in text.

### 2. Non-negative Matrix Factorization (NMF)
- Decomposes the TF-IDF matrix into smaller components.  
- Each component (topic) is described by the top keywords.  
- Works well for short and interpretable topics.

### 3. Latent Semantic Analysis (LSA)
- Uses **Singular Value Decomposition (SVD)** to identify underlying structures in text data.  
- Captures broader word associations but can be harder to interpret.



####**1) Latent Dirichlet Allocation (LDA)**


In [ ]:
# 1. Define the range of topics you want to test
topic_range = range(2, 11, 1)  # Tests 2 through 10
perplexity_scores = []

print("Training models to find the elbow...")

# 2. Loop through the range to calculate perplexity
for n in tqdm(topic_range):
    lda_model = LatentDirichletAllocation(n_components=n, random_state=42, n_jobs=-1)
    lda_model.fit(X_bow)
    # Perplexity: Lower is better
    perplexity_scores.append(lda_model.perplexity(X_bow))

# 3. Plot the Elbow Graph
plt.figure(figsize=(10, 6))
plt.plot(topic_range, perplexity_scores, marker='o', linestyle='-', color='b')
plt.title('Elbow Method: LDA Perplexity Score')
plt.xlabel('Number of Topics (n_components)')
plt.ylabel('Perplexity (Lower is Better)')
plt.xticks(topic_range)
plt.grid(True)
plt.show()


In [ ]:
# Set the number of topics
n_topics = 3

# Create and fit the LDA model
lda_model = LatentDirichletAllocation(n_components=n_topics, random_state=42)
lda_model.fit(X_bow)
#lda_model.fit(X_tfidf)

# Extract the top words from each topic
lda_topics = {}

for index, topic in enumerate(lda_model.components_):
    top_words = [feature_names[i] for i in topic.argsort()[-10:][::-1]]  # Get top 10 words and reverse order
    lda_topics[f"Topic {index+1}"] = top_words

# Print the LDA topics
print("LDA Topics:")
for topic, words in lda_topics.items():
    print(f"{topic}: {', '.join(words)}")



In [ ]:
# 1. Define your custom names based on your analysis
topic_names = {
    0: "Host & Hospitality",
    1: "Property & Amenities",
    2: "Strip & Location"
}

# 2. Updated function to plot top words with CUSTOM NAMES
def plot_top_words_renamed(model, feature_names, n_top_words, title, mapping):
    fig, axes = plt.subplots(1, n_topics, figsize=(15, 6), sharex=True)
    axes = axes.flatten()
    for topic_idx, topic in enumerate(model.components_):
        top_features_ind = topic.argsort()[: -n_top_words - 1 : -1]
        top_features = [feature_names[i] for i in top_features_ind]
        weights = topic[top_features_ind]

        ax = axes[topic_idx]
        ax.barh(top_features, weights, height=0.7)
        # Use the mapping dictionary for the title
        ax.set_title(mapping.get(topic_idx, f"Topic {topic_idx+1}"), fontdict={"fontsize": 14})
        ax.invert_yaxis()
        ax.tick_params(axis="both", which="major", labelsize=12)

    plt.suptitle(title, fontsize=20)
    plt.subplots_adjust(top=0.85, bottom=0.05, wspace=0.90)
    plt.show()

# 3. Updated function to plot word clouds with CUSTOM NAMES
def plot_word_clouds_renamed(model, feature_names, mapping):
    fig, axes = plt.subplots(1, n_topics, figsize=(15, 6))
    for topic_idx, topic in enumerate(model.components_):
        # Create a dictionary of word frequencies for the cloud
        word_freq = {feature_names[i]: topic[i] for i in range(len(feature_names))}
        wordcloud = WordCloud(background_color='white').generate_from_frequencies(word_freq)

        ax = axes[topic_idx]
        ax.imshow(wordcloud, interpolation='bilinear')
        # Use the mapping dictionary for the title
        ax.set_title(mapping.get(topic_idx, f"Topic {topic_idx+1}"))
        ax.axis('off')
    plt.show()

# --- EXECUTION ---

# Print text version
print("Renamed LDA Topics:")
for index, topic in enumerate(lda_model.components_):
    top_words = [feature_names[i] for i in topic.argsort()[-10:][::-1]]
    name = topic_names.get(index, f"Topic {index+1}")
    print(f"{name}: {', '.join(top_words)}")

# Generate Plots
plot_top_words_renamed(lda_model, feature_names, 10, 'Top words per LDA topic', topic_names)
plot_word_clouds_renamed(lda_model, feature_names, topic_names)

####**2)  Non-negative Matrix Factorization (NMF)**

In [ ]:
topic_range = range(2, 11, 1)  # Tests 2 through 10
nmf_errors = []

print("Training NMF models to find the elbow...")

# 2. Loop through the range to calculate reconstruction error
for n in tqdm(topic_range):
    nmf_model = NMF(n_components=n, random_state=42, init='nndsvd', max_iter=500)
    nmf_model.fit(X_bow) # You can also use X_tfidf here

    # Reconstruction Error: Lower is better
    nmf_errors.append(nmf_model.reconstruction_err_)

# 3. Plot the Elbow Graph
plt.figure(figsize=(10, 6))
plt.plot(topic_range, nmf_errors, marker='o', linestyle='-', color='g')
plt.title('Elbow Method: NMF Reconstruction Error')
plt.xlabel('Number of Topics (n_components)')
plt.ylabel('Reconstruction Error (Lower is Better)')
plt.xticks(topic_range)
plt.grid(True)
plt.show()

In [ ]:
# Create and fit the NMF model. 6 Topics as the curve starts to flatten afterwards
nmf_model = NMF(n_components=6, random_state=42, init='nndsvd', max_iter=500)
#nmf_model.fit(X_tfidf)
nmf_model.fit(X_bow)

# Extract the top words from each topic
nmf_topics = {}

for index, topic in enumerate(nmf_model.components_):
    top_words = [feature_names[i] for i in topic.argsort()[-10:][::-1]]  # Get top 10 words and reverse order
    nmf_topics[f"Topic {index+1}"] = top_words

# Print the NMF topics
print("NMF Topics:")
for topic, words in nmf_topics.items():
    print(f"{topic}: {', '.join(words)}")

In [ ]:
#Define NMF Topics
nmf_topic_names = {
    0: "Property & Amenities",
    1: "Overall Positive",
    2: "Clean & Recommended",
    3: "Memorable Stays",
    4: "Felt Like Home",
    5: "Strip & Location"
}

# 2. Print the NMF topics with the new names
print("Renamed NMF Topics:")
for index, topic in enumerate(nmf_model.components_):
    top_words = [feature_names[i] for i in topic.argsort()[-10:][::-1]]

    # Grab the name from our dictionary; fallback to "Topic X" if missing
    name = nmf_topic_names.get(index, f"Topic {index+1}")
    print(f"{name}: {', '.join(top_words)}")

In [ ]:
# Function for Bar Charts with Custom Names
def plot_top_words_nmf(model, feature_names, n_top_words, title, mapping):
    # Adjust layout based on 6 topics (2 rows of 3)
    fig, axes = plt.subplots(2, 3, figsize=(20, 10), sharex=True)
    axes = axes.flatten()
    for topic_idx, topic in enumerate(model.components_):
        top_features_ind = topic.argsort()[: -n_top_words - 1 : -1]
        top_features = [feature_names[i] for i in top_features_ind]
        weights = topic[top_features_ind]

        ax = axes[topic_idx]
        ax.barh(top_features, weights, height=0.7, color='skyblue')

        # KEY: Apply the custom name to the title
        ax.set_title(mapping.get(topic_idx, f"Topic {topic_idx+1}"), fontdict={"fontsize": 14})
        ax.invert_yaxis()
        ax.tick_params(axis="both", which="major", labelsize=10)

    plt.suptitle(title, fontsize=22)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

# Function for Word Clouds with Custom Names
def plot_nmf_word_clouds(model, feature_names, mapping):
    fig, axes = plt.subplots(2, 3, figsize=(15, 6))
    axes = axes.flatten()
    for topic_idx, topic in enumerate(model.components_):
        word_freq = {feature_names[i]: topic[i] for i in range(len(feature_names))}
        wordcloud = WordCloud(background_color='white', width=400, height=400).generate_from_frequencies(word_freq)

        ax = axes[topic_idx]
        ax.imshow(wordcloud, interpolation='bilinear')

        # KEY: Apply the custom name to the title
        ax.set_title(mapping.get(topic_idx, f"Topic {topic_idx+1}"), fontsize=16)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

# --- EXECUTION ---
plot_top_words_nmf(nmf_model, feature_names, 10, 'NMF Topics with Custom Labels', nmf_topic_names)
plot_nmf_word_clouds(nmf_model, feature_names, nmf_topic_names)

####**3) Latent Semantic Analysis (LSA)**

In [ ]:
topic_range = range(2, 11, 1)  # Tests 2 through 10
explained_variance = []

print("Training LSA (SVD) models to find the elbow...")

# 2. Loop through the range to calculate cumulative explained variance
for n in tqdm(topic_range):
    lsa_model = TruncatedSVD(n_components=n, random_state=42)
    lsa_model.fit(X_bow) # You can also use X_tfidf

    # We sum the variance of all components to see total info captured
    total_variance = lsa_model.explained_variance_ratio_.sum()
    explained_variance.append(total_variance)

# 3. Plot the Elbow Graph (Scree Plot)
plt.figure(figsize=(10, 6))
plt.plot(topic_range, explained_variance, marker='s', linestyle='-', color='r')
plt.title('LSA Elbow Method: Cumulative Explained Variance')
plt.xlabel('Number of Topics (n_components)')
plt.ylabel('Percent of Variance Explained (Higher is Better)')
plt.xticks(topic_range)
plt.grid(True)
plt.show()

In [ ]:
#Fit the LSA model. 4 topics (where curve starts to flatten)
lsa_model = TruncatedSVD(n_components=4, random_state=42)
lsa_model.fit(X_bow)

# Extract the top words for each topic
lsa_topics = {}

for index, topic in enumerate(lsa_model.components_):
    top_words = [feature_names[i] for i in topic.argsort()[-10:][::-1]]  # Get top 10 words and reverse order
    lsa_topics[f"Topic {index+1}"] = top_words

# Print the LSA topics
print("LSA Topics:")
for topic, words in lsa_topics.items():
    print(f"{topic}: {', '.join(words)}")

In [ ]:
#1. Define LSA topics
lsa_topic_names = {
    0: "Overall Positive Reviews",
    1: "Generic Praise",
    2: "Place Recommendations",
    3: "Memorable Stays"
}

# 2. Fit the LSA model (TruncatedSVD)
n_topics = 4
lsa_model = TruncatedSVD(n_components=n_topics, random_state=42)
lsa_model.fit(X_bow)

# 3. Print the Renamed Topics
print("Renamed LSA Topics:")
for index, topic in enumerate(lsa_model.components_):
    top_words = [feature_names[i] for i in topic.argsort()[-10:][::-1]]
    name = lsa_topic_names.get(index, f"Topic {index+1}")
    print(f"{name}: {', '.join(top_words)}")

# 4. Helper Function: Plot Top Words with Custom Names
def plot_lsa_top_words(model, feature_names, n_top_words, title, mapping):
    fig, axes = plt.subplots(2, 2, figsize=(20, 10), sharex=True)
    axes = axes.flatten()
    for topic_idx, topic in enumerate(model.components_):
        # LSA can have negative values, so we use the absolute value or just top positive
        top_features_ind = topic.argsort()[: -n_top_words - 1 : -1]
        top_features = [feature_names[i] for i in top_features_ind]
        weights = topic[top_features_ind]

        ax = axes[topic_idx]
        ax.barh(top_features, weights, height=0.7, color='salmon')
        ax.set_title(mapping.get(topic_idx, f"Topic {topic_idx+1}"), fontsize=14)
        ax.invert_yaxis()
        ax.tick_params(axis="both", which="major", labelsize=10)

    plt.suptitle(title, fontsize=22)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.show()

# 5. Helper Function: Plot Word Clouds with Custom Names
def plot_lsa_word_clouds(model, feature_names, mapping):
    fig, axes = plt.subplots(2, 2, figsize=(15, 6))
    axes = axes.flatten()
    for topic_idx, topic in enumerate(model.components_):
        # LSA components can be negative; word clouds require positive values
        # We take the absolute value of the weights for visualization purposes
        word_freq = {feature_names[i]: abs(topic[i]) for i in range(len(feature_names))}
        wordcloud = WordCloud(background_color='white', width=400, height=400).generate_from_frequencies(word_freq)

        ax = axes[topic_idx]
        ax.imshow(wordcloud, interpolation='bilinear')
        ax.set_title(mapping.get(topic_idx, f"Topic {topic_idx+1}"), fontsize=16)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

# --- 6. Generate the Visuals ---
plot_lsa_top_words(lsa_model, feature_names, 10, 'LSA Topics: Top Words per Component', lsa_topic_names)
plot_lsa_word_clouds(lsa_model, feature_names, lsa_topic_names)

**Topic Distance**

In [ ]:
#Check the topic distance

# Function to plot topic distances using PCA
def plot_topic_distance_pca(model, title='Topic Distance (PCA)'):
    # Get the topic-word matrix (components_) from the model
    topic_word_matrix = model.components_

    # Apply PCA to reduce the topic-word matrix to 2 dimensions
    pca = PCA(n_components=2)
    topic_pca = pca.fit_transform(topic_word_matrix)

    # Plot the topics in 2D space
    plt.figure(figsize=(8, 6))
    plt.scatter(topic_pca[:, 0], topic_pca[:, 1], c='blue', edgecolor='k', s=100)

    # Annotate the topics with text labels
    for i in range(len(topic_pca)):
        plt.text(topic_pca[i, 0], topic_pca[i, 1], f'Topic {i+1}', fontsize=12)

    plt.title(title)
    plt.xlabel('PCA Component 1')
    plt.ylabel('PCA Component 2')
    plt.show()

In [ ]:
# compare topic distance of all models

# LDA
plot_topic_distance_pca(lda_model, title='LDA Topic Distance (PCA)')
# NMF
plot_topic_distance_pca(nmf_model, title='NMF Topic Distance (PCA)')
# LSA
plot_topic_distance_pca(lsa_model, title='LSA Topic Distance (PCA)')


### Step 4: Modern Topic Modeling with BERTopic

Unlike the traditional approaches above, BERTopic uses transformer-based embeddings
to capture the *semantic meaning* of text rather than just word frequencies. This
means it can recognize that "clean" and "tidy" refer to the same concept, rather
than treating them as unrelated words.

**How it works:**
- Each review is converted into a dense vector embedding using a pre-trained
sentence transformer model
- Similar reviews are grouped together through clustering
- Topics are defined by the most representative terms in each cluster

**Key advantages over LDA/NMF/LSA:**
- Context-aware — understands word meaning, not just frequency
- Does not require specifying the number of topics upfront
- Generally produces more semantically meaningful and distinct topics

**Model used:** BERTopic defaults to `all-MiniLM-L6-v2`, a lightweight
sentence transformer from Hugging Face that balances speed and accuracy well
for this use case.

In [ ]:
# Sample a subset of reviews for BERTopic - transformer-based models are memory intensive
sampled_reviews = filtered_reviews.sample(n=500, random_state=42)

# Prepare the text data for BERTopic
texts = sampled_reviews['cleaned_comments'].tolist()

# Fit the BERTopic model
topic_model = BERTopic()
topics, probabilities = topic_model.fit_transform(texts)

# Get topic information
topic_info = topic_model.get_topic_info()
print(topic_info)

In [ ]:
# Visualize the word distribution for the topics
topic_model.visualize_barchart()

In [ ]:
# Visualize the hierarchical clustering of topics
topic_model.visualize_hierarchy()

In [ ]:
# Visualize the topic similarity using a heatmap
topic_model.visualize_heatmap()

In [ ]:
topic_model.visualize_topics()

## Summary & Key Findings

**Price Prediction:** A feed-forward neural network (TensorFlow/Keras, 64→32 dense layers with dropout regularization) achieved R² of 0.504, MAE of \$58.50, and RMSE of \$95.00 on the held-out test set after capping prices at \$1,000/night to exclude luxury rentals. Permutation importance revealed that capacity features dominate predictions — bathrooms (importance: 0.30) and bedrooms (0.20) together drive most of the model's predictive power, with room type adding additional signal. Notably, location-specific features (latitude, longitude, individual neighborhoods) contributed almost nothing, suggesting that for Vegas Airbnbs, the type and size of a listing matters more than its specific location.

**Review Topic Modeling:** LDA, NMF, LSA, and BERTopic produced complementary views at different granularities. LDA identified three broad themes, host hospitality, property features, and Strip-area location while NMF refined these into six more granular sub-themes (including a generic-praise component capturing common positive language). LSA underperformed, with three of four components capturing generic praise patterns rather than distinct themes, illustrating its known weakness on short, lexically repetitive review text. BERTopic provided a complementary semantic clustering view.

**Tools used:** Python, pandas, NumPy, scikit-learn, TensorFlow/Keras, NLTK, BERTopic, Seaborn, Matplotlib, Plotly